In [0]:

# Step 1 - Read the Weather Parquet File

# Replace the file path if your file name is slightly different.

from pyspark.sql.functions import *
from pyspark.sql.types import *

spark.sql("USE CATALOG Bronze_Catalog")
spark.sql("USE SCHEMA Bronze_SCH")

weather_df = spark.read.parquet(
    "abfss://pro2@adlsstgacntp08.dfs.core.windows.net/parquet/weather_stream_v2.parquet")

In [0]:
# Step 2 - Verify the Data
display(weather_df)

In [0]:
# Step 3 - Check Schema
weather_df.printSchema()

In [0]:
# Step 4 - Convert Timestamp
from pyspark.sql.functions import col, to_timestamp

weather_df = weather_df.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "dd-MM-yyyy")
)

In [0]:
last_watermark = spark.sql("""
SELECT last_processed_timestamp
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
WHERE table_name='Bronze_Weather'
""").collect()[0][0]

print(last_watermark)

In [0]:
incremental_df = weather_df.filter(
    col("timestamp") > last_watermark
)

records_loaded = incremental_df.count()

print(f"New Records : {records_loaded}")

display(incremental_df.limit(10))

In [0]:
from pyspark.sql.functions import current_timestamp

try:

    incremental_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema","true") \
        .saveAsTable("Bronze_Catalog.Bronze_SCH.Bronze_Weather")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Weather',
            'Weather Bronze Pipeline',
            current_timestamp(),
            {records_loaded},
            'SUCCESS',
            NULL
        )
    """)

    print(f"✅ Bronze_Weather loaded successfully.....Records Loaded : {records_loaded}")
    

except Exception as e:

    error_message = str(e).replace("'", "''")

    spark.sql(f"""
        INSERT INTO Bronze_Catalog.Bronze_SCH.Audit_Log
        VALUES(
            'Bronze_Weather',
            'Weather Bronze Pipeline',
            current_timestamp(),
            0,
            'FAILED',
            '{error_message}'
        )
    """)

    print("❌ Bronze_Weather Pipeline Failed")
    print(f"⚠️ Error : {error_message}")
    
    

    raise

In [0]:
spark.sql("""
UPDATE Bronze_Catalog.Bronze_SCH.Watermark_Table
SET last_processed_timestamp = (
    SELECT MAX(timestamp)
    FROM Bronze_Catalog.Bronze_SCH.Bronze_Weather
)
WHERE table_name='Bronze_Weather'
""")

print("✅ Watermark Updated Successfully")

In [0]:
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Watermark_Table
WHERE table_name='Bronze_Weather'
""").show(truncate=False)

In [0]:
spark.sql("""
SELECT *
FROM Bronze_Catalog.Bronze_SCH.Audit_Log
WHERE table_name='Bronze_Weather'
ORDER BY load_time DESC
LIMIT 5
""").show(truncate=False)